# Bonus B1 solution, notebook 1: show that the restart is not the cause of the second slip, and answer a slip with no

The worked answers to the two challenge tasks of `b1a_meet_durable_workflows.ipynb`. The setup repeats the
notebook's state, nodes and both graphs, so this file runs on its own, in one kernel, with no restart. It calls
no model and needs no key.

## Setup

In [1]:
import sqlite3
from pathlib import Path
from typing_extensions import TypedDict

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
TRIP_FILE = OUTPUTS / "trip.db"

for old_file in OUTPUTS.glob("trip.db*"):
    old_file.unlink()

saver = SqliteSaver(sqlite3.connect(TRIP_FILE, check_same_thread=False))

The state and the nodes. `ask_parent` sends the slip and waits in one node (Parts A to D). `send_slip` and
`wait_for_parent` are the same work split in two (Part E).

In [2]:
class Slip(TypedDict):
    child: str
    trip: str
    allowed: bool


def ask_parent(slip):
    print("SLIP SENT HOME with", slip["child"])
    answer = interrupt(f"May {slip['child']} go to {slip['trip']}?")
    return {"allowed": answer == "yes"}


def send_slip(slip):
    print("SLIP SENT HOME with", slip["child"])
    return {}


def wait_for_parent(slip):
    answer = interrupt(f"May {slip['child']} go to {slip['trip']}?")
    return {"allowed": answer == "yes"}


def book_seat(slip):
    if slip["allowed"]:
        print("SEAT BOOKED for", slip["child"])
    else:
        print("NO SEAT for", slip["child"])
    return {}

The two graphs, on one checkpoint file: `graph` with the one node that sends and waits, and `graph_v2` with the
two nodes.

In [3]:
builder = StateGraph(Slip)
builder.add_node("ask_parent", ask_parent)
builder.add_node("book_seat", book_seat)
builder.add_edge(START, "ask_parent")
builder.add_edge("ask_parent", "book_seat")
builder.add_edge("book_seat", END)
graph = builder.compile(checkpointer=saver)

builder = StateGraph(Slip)
builder.add_node("send_slip", send_slip)
builder.add_node("wait_for_parent", wait_for_parent)
builder.add_node("book_seat", book_seat)
builder.add_edge(START, "send_slip")
builder.add_edge("send_slip", "wait_for_parent")
builder.add_edge("wait_for_parent", "book_seat")
builder.add_edge("book_seat", END)
graph_v2 = builder.compile(checkpointer=saver)

print(list(graph.get_graph().nodes))
print(list(graph_v2.get_graph().nodes))

['__start__', 'ask_parent', 'book_seat', '__end__']
['__start__', 'send_slip', 'wait_for_parent', 'book_seat', '__end__']


## Task 1: Show that the restart is not the cause

Sam's slip goes out on the first graph, under a thread id of its own.

In [4]:
sam = {"configurable": {"thread_id": "slip-sam"}}

paused = graph.invoke({"child": "Sam", "trip": "the science museum"}, sam)

print("question:", paused["__interrupt__"][0].value)

SLIP SENT HOME with Sam
question: May Sam go to the science museum?


One `SLIP SENT HOME with Sam` line, and the run waits. We answer in the same kernel. Nothing was restarted, and
`graph` is the same object.

In [5]:
done = graph.invoke(Command(resume="yes"), sam)

print("result:", done)

SLIP SENT HOME with Sam
SEAT BOOKED for Sam
result: {'child': 'Sam', 'trip': 'the science museum', 'allowed': True}


`SLIP SENT HOME with Sam` was printed again, by the resume, and then `SEAT BOOKED for Sam`. The slip went home
twice with no restart in between.

So the second slip does not come from the restart. It comes from the resume: LangGraph runs the waiting node
again from its first line every time, because a checkpoint records which node is next and not which line it
reached. The restart in the notebook only removed every other explanation.

## Task 2: Answer a slip with no

Noor's slip goes out on `graph_v2`, where sending and waiting are two nodes.

In [6]:
noor = {"configurable": {"thread_id": "slip-noor"}}

paused = graph_v2.invoke({"child": "Noor", "trip": "the science museum"}, noor)

print("question: ", paused["__interrupt__"][0].value)
print("next node:", graph_v2.get_state(noor).next)

SLIP SENT HOME with Noor
question:  May Noor go to the science museum?
next node: ('wait_for_parent',)


The parent says no. We send a no the same way as a yes: `Command(resume=...)` with the same `config`.

In [7]:
done = graph_v2.invoke(Command(resume="no"), noor)

print("result:   ", done)
print("next node:", graph_v2.get_state(noor).next)

NO SEAT for Noor
result:    {'child': 'Noor', 'trip': 'the science museum', 'allowed': False}
next node: ()


`NO SEAT for Noor`, and `allowed` is `False`. `wait_for_parent` compared the answer with `"yes"`, saved `False`,
and `book_seat` read it. The slip was sent once: the resume printed no `SLIP SENT HOME` line, because
`send_slip` had finished before the wait. `next` is empty, so the thread is finished, without a seat.

The graph does not treat an answer as a permission. Our own line `answer == "yes"` decides what the answer
means.

Back to [the notebook](../b1a_meet_durable_workflows.ipynb).